# Подготовка данных о рынке видеоигр

**Автор:** Алексей Поляков  
**Период исследования:** 2000–2013 годы

Подготовка аналитического среза исторических данных о видеоиграх: очистка данных, обработка пропусков и дубликатов, категоризация оценок и определение наиболее представленных платформ.


## Цель

Подготовить корректный набор данных для последующего исследования развития игровой индустрии, продаж по регионам и жанрам, включая отдельный анализ RPG.

В рамках текущего этапа:

- проверяется качество исходных данных;
- исправляются типы и формат полей;
- обрабатываются пропуски и дубликаты;
- формируется срез за 2000–2013 годы;
- оценки пользователей и критиков распределяются по категориям;
- определяется топ-7 платформ по количеству выпущенных игр.


## 1. Загрузка и первичный обзор

In [1]:
from pathlib import Path
try:
    from IPython.display import display
except ImportError:
    display = print
import numpy as np
import pandas as pd

local_paths = [Path('../data/new_games.csv'), Path('data/new_games.csv')]
data_path = next((path for path in local_paths if path.exists()), None)

if data_path is not None:
    df = pd.read_csv(data_path)
else:
    df = pd.read_csv('https://code.s3.yandex.net/datasets/new_games.csv')

initial_rows = len(df)
display(df.head())
df.info()

                       Name Platform  ...  User Score Rating
0                Wii Sports      Wii  ...           8      E
1         Super Mario Bros.      NES  ...         NaN    NaN
2            Mario Kart Wii      Wii  ...         8.3      E
3         Wii Sports Resort      Wii  ...           8      E
4  Pokemon Red/Pokemon Blue       GB  ...         NaN    NaN

[5 rows x 11 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   floa

Исходный датасет содержит 16 956 строк и 11 полей. Пропуски присутствуют прежде всего в оценках пользователей, оценках критиков и рейтинге ESRB. Поля `eu_sales`, `jp_sales` и `user_score` требуют преобразования в числовой формат, а названия столбцов — приведения к `snake_case`.

## 2. Предобработка

In [2]:
df.columns = df.columns.str.replace(' ', '_', regex=False).str.lower()

for column in ['eu_sales', 'jp_sales', 'user_score']:
    df[column] = pd.to_numeric(
        df[column].astype('string').str.replace(',', '.', regex=False),
        errors='coerce'
    )

missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_share_pct': (df.isna().mean() * 100).round(2)
}).sort_values('missing_share_pct', ascending=False)

display(missing)

                 missing_count  missing_share_pct
user_score                9268              54.66
critic_score              8714              51.39
rating                    6871              40.52
year_of_release            275               1.62
eu_sales                     6               0.04
jp_sales                     4               0.02
name                         2               0.01
genre                        2               0.01
platform                     0               0.00
na_sales                     0               0.00
other_sales                  0               0.00


Оценки и рейтинг ESRB не восстанавливаются искусственно: их замена средним или модой исказила бы дальнейшие сравнения. Строки без названия или года выпуска не позволяют идентифицировать игру и отнести её к исследуемому периоду, поэтому удаляются.

In [3]:
df = df.dropna(subset=['name', 'year_of_release']).copy()
df['year_of_release'] = df['year_of_release'].astype('int64')

for column in ['na_sales', 'eu_sales', 'jp_sales']:
    group_mean = df.groupby(['platform', 'year_of_release'])[column].transform('mean')
    df[column] = df[column].fillna(group_mean)

for column in ['name', 'platform', 'genre']:
    df[column] = df[column].str.lower().str.strip()
df['rating'] = df['rating'].str.upper().str.strip()

duplicate_rows = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
removed_rows = initial_rows - len(df)

print(f'Удалено полных дубликатов: {duplicate_rows}')
print(f'Всего удалено строк: {removed_rows} ({removed_rows / initial_rows:.2%})')
print(f'Размер подготовленного датасета: {df.shape[0]:,} строк × {df.shape[1]} столбцов')

Удалено полных дубликатов: 235
Всего удалено строк: 512 (3.02%)
Размер подготовленного датасета: 16,444 строк × 11 столбцов


После очистки осталось 16 444 строки. Всего удалено 512 строк — около 3,02% исходного набора; в том числе 235 полных дубликатов.

## 3. Срез за 2000–2013 годы

In [4]:
df_actual = df.loc[df['year_of_release'].between(2000, 2013)].copy()

print(f'Размер аналитического среза: {len(df_actual):,} строк')
print(f'Период: {df_actual["year_of_release"].min()}–{df_actual["year_of_release"].max()}')
display(df_actual.head())

Размер аналитического среза: 12,781 строк
Период: 2000–2013
                    name platform  ...  user_score rating
0             wii sports      wii  ...         8.0      E
2         mario kart wii      wii  ...         8.3      E
3      wii sports resort      wii  ...         8.0      E
6  new super mario bros.       ds  ...         8.5      E
7               wii play      wii  ...         6.6      E

[5 rows x 11 columns]


В итоговый временной срез вошла 12 781 запись об играх, выпущенных с 2000 по 2013 год включительно.

## 4. Категоризация оценок

In [5]:
def categorize_score(series, medium_boundary, high_boundary):
    conditions = [
        series.lt(medium_boundary).fillna(False).to_numpy(dtype=bool),
        (series.ge(medium_boundary) & series.lt(high_boundary)).fillna(False).to_numpy(dtype=bool),
        series.ge(high_boundary).fillna(False).to_numpy(dtype=bool),
    ]
    labels = ['низкая оценка', 'средняя оценка', 'высокая оценка']
    return np.select(conditions, labels, default='без оценки')


df_actual['user_score_category'] = categorize_score(
    df_actual['user_score'], medium_boundary=3, high_boundary=8
)
df_actual['critic_score_category'] = categorize_score(
    df_actual['critic_score'], medium_boundary=30, high_boundary=80
)

category_summary = pd.concat(
    [
        df_actual['user_score_category'].value_counts().rename('user_score'),
        df_actual['critic_score_category'].value_counts().rename('critic_score'),
    ],
    axis=1,
).fillna(0).astype(int)

display(category_summary)

                user_score  critic_score
без оценки            6298          5612
средняя оценка        4081          5422
высокая оценка        2286          1692
низкая оценка          116            55


Категории сформированы независимо для каждого типа оценки. Пропуски явно отмечены как `без оценки`, что позволяет не смешивать отсутствие данных с низкими баллами.

## 5. Топ-7 платформ

In [6]:
top_platforms = (
    df_actual.groupby('platform')['name']
    .nunique()
    .sort_values(ascending=False)
    .head(7)
    .rename('games_count')
    .reset_index()
)

display(top_platforms)

  platform  games_count
0      ps2         2127
1       ds         2120
2      wii         1275
3      psp         1180
4     x360         1120
5      ps3         1086
6      gba          811


По количеству уникальных игр лидируют **PlayStation 2 (2 127)** и **Nintendo DS (2 120)**. Далее следуют Wii, PSP, Xbox 360, PlayStation 3 и Game Boy Advance. Этот результат описывает представленность платформ в датасете, а не объём продаж или коммерческую эффективность.

## Итоги

- Названия полей приведены к `snake_case`, числовые признаки — к корректным типам.
- Проанализированы и обработаны пропуски; оценки и ESRB-рейтинг оставлены без искусственного заполнения.
- Удалены строки без ключевой информации и 235 полных дубликатов; суммарно отброшено около 3,02% исходных строк.
- Подготовлен срез из 12 781 записи за 2000–2013 годы.
- Добавлены признаки `user_score_category` и `critic_score_category`.
- Определены семь платформ с наибольшим количеством уникальных игр.

Подготовленный срез можно использовать на следующем этапе для анализа продаж по регионам и жанрам, динамики платформ и отдельного исследования RPG. Такие выводы не делаются в текущей работе, поскольку её задача ограничена подготовкой данных.
